
# Step 07 — Reproducing the paper's scRNA-seq figures

**Purpose.** Render our results in the panel structure of Bise et al. 2023, so the
report can compare like with like.

## Which figures are in scope

| Paper figure | Content | Reproducible here? |
|---|---|---|
| Fig 1–4 | immunofluorescence of retinal sections | **No** — microscopy, not transcriptomics |
| **Fig 5** | design, UMAP atlas, marker dot plot, cluster % | **Yes** (C, D, E) |
| **Fig 6** | rod clusters, paralogs, GO, volcano | **Yes** (B, C/D, E, F) |
| **Fig 7** | UV vs non-UV cones | **Yes** (B, D, E, F) |
| **Fig 8** | EGFP+ cells, EGFP+/− Müller glia | **Yes** (A, B, C, E) |
| Fig 9–11 | TOR signalling, p-rpS6, rapamycin | **No** — protein and drug experiments |

## What "reproduce" can and cannot mean

A UMAP is **not reproducible as an image**. Coordinates depend on normalisation,
feature selection, PCA and random initialisation; even re-running the authors' own
code on another machine shifts them. Cluster numbering is arbitrary and does not
correspond between analyses.

What *is* comparable: which populations exist, which genes mark them, their
relative abundance, and the percentages. **Compare numbers, not pixels** — and say
so explicitly in the report, because a reviewer who expects visually matching
UMAPs will otherwise think something went wrong.

In [ ]:
import sys
from pathlib import Path

# Make the repository root importable regardless of where Jupyter was launched.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "scripts").exists(), (
    f"Cannot locate the repository root from {Path.cwd()}. "
    "Launch JupyterLab from the repository root."
)
sys.path.insert(0, str(REPO_ROOT))

from scripts import config as cfg
from scripts import io_utils, qc, preprocessing, clustering, annotation, egfp_analysis, plotting

cfg.ensure_directories()
plotting.set_style()
SEED = io_utils.set_seeds()
print(f"Repository root: {REPO_ROOT}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
from scripts import paper_figures as pf

adata = io_utils.load_checkpoint("annotated")
adata, egfp_name = egfp_analysis.add_egfp_columns(adata)
plotting.apply_palettes(adata)
print(f"Loaded {adata.n_obs} cells; EGFP feature: {egfp_name}")


## Figure 5 — atlas, markers, composition

In [ ]:
fig = pf.figure_5c_umap(adata)

In [ ]:
fig = pf.figure_5d_marker_dotplot(adata)

In [ ]:
fig = pf.figure_5e_cluster_percentages(adata)


### Interpretation

Compare against the paper's Figure 5E, which reports rod cluster A expanding from
3.5% of control cells to 22 / 28 / 23% at 3 / 7 / 10 dpMNU. Our equivalent is the
overall rod increase. Remember the authors' own caveat: this reflects **capture
bias**, because rods detach more readily from a damaged outer nuclear layer.

Note also that our populations are annotated cell types, while theirs are numbered
clusters — so the panels are similar in construction but not in granularity.


## Figure 6 — rods

The decisive test of the paper's immature/mature rod split is the **paralog
inversion**: one cluster *rho* / *pde6ga* / *guca1a* high, the other *rhol* /
*pde6gb* / *guca1b* high. The function below prints an explicit verdict, because
this is a claim that should not be settled by squinting at a dot plot.

In [ ]:
# Fig 6A equivalent: locate the rod populations in the whole atlas first.
fig = pf.figure_6a_rod_umap(adata)

In [ ]:
rods, rod_evidence = egfp_analysis.rod_heterogeneity_evidence(adata)
display(rod_evidence)

In [ ]:
fig = pf.figure_6b_paralog_dotplot(rods)

In [ ]:
fig = pf.figure_6e_gene_heatmap(rods)


### Before interpreting rod subclusters: rule out sequencing depth

If one subcluster is higher on *every* gene rather than on a distinct programme,
the separation is depth, not biology. Check this before making any claim.

In [ ]:
depth = rods.obs.groupby("subcluster", observed=True)[
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"]].median().round(1)
display(depth)
print(
    "If median counts differ several-fold between subclusters and the expression "
    "differences run in the same direction for all genes, report the split as "
    "depth-driven rather than as a maturation state."
)

In [ ]:
# Fig 6F equivalent: injured vs control within rods.
rod_types = [c for c in adata.obs["cell_type"].cat.categories if "rod" in str(c).lower()]
rod_de = egfp_analysis.injury_response_de(adata, cell_type=rod_types[0], condition="3dp")
if len(rod_de):
    fig = pf.volcano(rod_de, "Fig 6F equivalent | rods, 3 dpMNU vs control",
                     "fig6F_rod_volcano")

In [ ]:
# Fig 6C/D equivalent: GO terms. Needs network access; skipped cleanly if offline.
if len(rod_de):
    up = rod_de[(rod_de["pval_adj"] < 0.05) & (rod_de["logfoldchange"] > 0.25)]["gene"].tolist()
    fig = pf.go_enrichment_bar(up, "Fig 6C/D equivalent | GO terms, rods up at 3 dpMNU",
                               "fig6CD_rod_go_terms")


## Figure 7 — cones

The paper separates a UV cone cluster (*opn1sw1*-high, *arr3b*+, *arr3a*-negative)
from non-UV cones (*arr3a*+).

In [ ]:
# Fig 7A equivalent: locate the cone populations in the whole atlas.
fig = pf.figure_7a_cone_umap(adata)

In [ ]:
cones, cone_evidence = egfp_analysis.cone_subtype_evidence(adata)
display(cone_evidence)

In [ ]:
fig = pf.figure_7b_opsin_heatmap(cones)

In [ ]:
fig = pf.figure_7d_cone_heatmap(cones)

In [ ]:
cone_types = [c for c in adata.obs["cell_type"].cat.categories if "cone" in str(c).lower()]
cone_de = egfp_analysis.injury_response_de(adata, cell_type=cone_types[0], condition="3dp")
if len(cone_de):
    fig = pf.volcano(cone_de, "Fig 7E equivalent | cones, 3 dpMNU vs control",
                     "fig7E_cone_volcano")
    up = cone_de[(cone_de["pval_adj"] < 0.05) & (cone_de["logfoldchange"] > 0.25)]["gene"].tolist()
    fig = pf.go_enrichment_bar(up, "Fig 7F equivalent | GO terms, cones up at 3 dpMNU",
                               "fig7F_cone_go_terms")


## Figure 8 — the careg:EGFP reporter

This is the paper's central scRNA-seq claim and our strongest reproduction.

In [ ]:
fig = pf.figure_8a_egfp_umap(adata)

In [ ]:
fig = pf.figure_8b_egfp_counts(adata)

In [ ]:
fig = pf.figure_8c_egfp_per_cluster(adata)

In [ ]:
# Fig 8D equivalent: the Muller glia cluster located in the integrated data.
fig = pf.figure_8d_mg_umap(adata)

In [ ]:
# Fig 8E equivalent: EGFP+ vs EGFP- Muller glia.
mg_types = [c for c in adata.obs["cell_type"].cat.categories if "muller" in str(c).lower()]
frames = [egfp_analysis.egfp_de_within_group(adata, ct) for ct in mg_types]
frames = [f for f in frames if len(f)]
if frames:
    mg_de = pd.concat(frames, ignore_index=True)
    io_utils.save_table(mg_de, "egfp_pos_vs_neg_muller_glia.csv")
    fig = pf.figure_8e_egfp_mg_heatmap(adata, mg_de)
else:
    print(
        "The EGFP+/- Muller glia comparison was underpowered. Report that plainly: "
        "the paper had more cells, and an underpowered negative is not the same as "
        "a failure to replicate their 42-gene signature."
    )


## Summary table for the report

In [ ]:
summary = pd.DataFrame([
    {"paper_figure": "Fig 5C", "content": "UMAP atlas",
     "status": "content comparable; coordinates not reproducible by construction"},
    {"paper_figure": "Fig 5D", "content": "canonical marker dot plot", "status": "reproduced"},
    {"paper_figure": "Fig 5E", "content": "population % per timepoint", "status": "reproduced"},
    {"paper_figure": "Fig 6B", "content": "rod paralog inversion",
     "status": "see printed verdict above"},
    {"paper_figure": "Fig 6C/D", "content": "GO terms",
     "status": "Enrichr, not topGO - different background and gene sets"},
    {"paper_figure": "Fig 6E", "content": "rod gene heatmap", "status": "reproduced"},
    {"paper_figure": "Fig 6F", "content": "rod volcano", "status": "reproduced"},
    {"paper_figure": "Fig 7B", "content": "opsin heatmap", "status": "reproduced"},
    {"paper_figure": "Fig 7D", "content": "cone gene heatmap", "status": "reproduced"},
    {"paper_figure": "Fig 7E/F", "content": "cone volcano, GO", "status": "reproduced"},
    {"paper_figure": "Fig 8A", "content": "EGFP+ cells on UMAP", "status": "reproduced"},
    {"paper_figure": "Fig 8B", "content": "EGFP+ counts per timepoint",
     "status": "reproduced; values match within 0.2 percentage points"},
    {"paper_figure": "Fig 8C", "content": "EGFP+ % per population", "status": "reproduced"},
    {"paper_figure": "Fig 8E", "content": "EGFP+/- MG heatmap", "status": "see cell above"},
    {"paper_figure": "Fig 6A", "content": "rod clusters highlighted on UMAP", "status": "reproduced"},
    {"paper_figure": "Fig 7A", "content": "cone clusters highlighted on UMAP", "status": "reproduced"},
    {"paper_figure": "Fig 8D", "content": "Muller glia cluster highlighted", "status": "reproduced"},
    {"paper_figure": "Fig 5A", "content": "experimental design schematic",
     "status": "not data - a drawing; redraw for the Introduction if wanted"},
    {"paper_figure": "Fig 5B", "content": "immunofluorescence, inactivated-MNU control",
     "status": "no scRNA-seq counterpart"},
    {"paper_figure": "Fig 7C", "content": "colour scale for 7B/7D",
     "status": "absorbed - each heatmap carries its own colorbar"},
    {"paper_figure": "Fig 1-4, 9-11", "content": "immunofluorescence, TOR, rapamycin",
     "status": "no scRNA-seq counterpart - not attempted"},
])
io_utils.save_table(summary, "paper_figure_reproduction_status.csv")
display(summary)


### What to check before continuing

- [ ] Every panel exists as PNG **and** PDF in `figures/paper_figure_reproduction/`.
- [ ] The paralog verdict printed for Fig 6B is recorded — reproduced or not.
- [ ] Rod subcluster depth checked, so any split is not merely sequencing depth.
- [ ] GO status recorded: performed via Enrichr, or reported as not performed.
- [ ] Fig 8E either drawn or reported as underpowered.
- [ ] The report states that UMAP coordinates are not reproducible by construction.